In [18]:
import pandas as pd
from owlready2 import *
import urllib.parse


In [19]:
# Định nghĩa không gian tên (Namespace)
BASE_IRI = "http://www.semanticweb.org/phandangvu/ontologies/2026/8/v-idiomv2#"
onto = get_ontology(BASE_IRI)

with onto:
    # --- Định nghĩa các Lớp (Classes) ---
    class Thành_ngữ_Tục_ngữ(Thing): pass
    
    class Vietnamese_idiom(Thành_ngữ_Tục_ngữ): pass
    class English_Idiom(Thành_ngữ_Tục_ngữ): pass
    
    class Tiêu_chí(Thing): pass
    
    class Bối_cảnh(Tiêu_chí): pass
    class Hành_động(Tiêu_chí): pass
    class Kết_quả(Tiêu_chí): pass
    class Mục_đích(Tiêu_chí): pass
    
    class Khung_Bản_Thể_học(Thing): pass
    
        # --- Định nghĩa các Thuộc tính (Object Properties) ---
    class Có_bối_cảnh(ObjectProperty):
        domain    = [Khung_Bản_Thể_học]
        range     = [Bối_cảnh]
        
    class Có_hành_động(ObjectProperty):
        domain    = [Khung_Bản_Thể_học]
        range     = [Hành_động]
        
    class Có_kết_quả(ObjectProperty):
        domain    = [Khung_Bản_Thể_học]
        range     = [Kết_quả]
        
    class Có_mục_đích(ObjectProperty):
        domain    = [Khung_Bản_Thể_học]
        range     = [Mục_đích]
        
    # Định nghĩa quan hệ 2 chiều giữa Idiom và Khung
    class Là_khung_của(ObjectProperty):
        domain    = [Khung_Bản_Thể_học]
        range     = [Thành_ngữ_Tục_ngữ]

    class Có_khung(ObjectProperty):
        domain    = [Thành_ngữ_Tục_ngữ]
        range     = [Khung_Bản_Thể_học]
        inverse_property = Là_khung_của
        
    # Thuộc tính suy luận: Có_khung ∘ Là_khung_của -> Có_nghĩa
    class Có_nghĩa(ObjectProperty):
        domain    = [Thành_ngữ_Tục_ngữ]
        range     = [Thành_ngữ_Tục_ngữ]
        property_chain = [PropertyChain([Có_khung, Là_khung_của])]



In [20]:
# Đọc file CSV chứa 100 câu thành ngữ/tục ngữ V3
csv_path = '../../thanh_ngu_tuc_ngu_100_cau_v4.csv'
df = pd.read_csv(csv_path)

# Điền các ô trống bằng chuỗi rỗng để dễ xử lý (không gán bừa tiêu chí)
df = df.fillna('')

# Lấy danh sách các thuộc tính duy nhất để tạo Individuals cho Tiêu chí
bctx = df['Bối_cảnh'].unique()
hdtx = df['Hành_động'].unique()
kqtx = df['Kết_quả'].unique()
mdtx = df['Mục_đích'].unique()

# Hàm tạo Individual cho tiêu chí
def create_criteria_instances(cls, items):
    with onto:
        for item in items:
            item = str(item).strip()
            if item != '':
                # Owlready2 tự động đăng ký individual
                cls(item)

create_criteria_instances(onto.Bối_cảnh, bctx)
create_criteria_instances(onto.Hành_động, hdtx)
create_criteria_instances(onto.Kết_quả, kqtx)
create_criteria_instances(onto.Mục_đích, mdtx)

print("Đã tạo xong các cá thể Tiêu chí!")


Đã tạo xong các cá thể Tiêu chí!


In [21]:
# Quản lý các khung đã tạo (dictionary để tránh trùng lặp)
frame_dict = {}

with onto:
    for idx, row in df.iterrows():
        # Xử lý tên thành ngữ (bỏ khoảng trắng thừa nếu có)
        vn_name = str(row['Thành_ngữ_Tiếng_Việt']).strip().replace(' ', '_')
        en_name = str(row['Thành_ngữ_Tiếng_Anh']).strip().replace(' ', '_')
        
        # Tiêu chí
        bc = str(row['Bối_cảnh']).strip()
        hd = str(row['Hành_động']).strip()
        kq = str(row['Kết_quả']).strip()
        md = str(row['Mục_đích']).strip()
        
        # Tạo signature cho Khung
        frame_key = f"{bc}|{hd}|{kq}|{md}"
        
        if frame_key not in frame_dict:
            # Tạo khung mới
            frame_name = f"Khung{len(frame_dict) + 1}"
            new_frame = onto.Khung_Bản_Thể_học(frame_name)
            
            # Gán tiêu chí cho khung
            if bc != '': new_frame.Có_bối_cảnh.append(onto[bc])
            if hd != '': new_frame.Có_hành_động.append(onto[hd])
            if kq != '': new_frame.Có_kết_quả.append(onto[kq])
            if md != '': new_frame.Có_mục_đích.append(onto[md])
                
            frame_dict[frame_key] = new_frame
            
        current_frame = frame_dict[frame_key]
        
        # Tạo individual cho thành ngữ Việt và Anh
        if vn_name != '':
            vn_inst = onto.Vietnamese_idiom(vn_name)
            vn_inst.Có_khung.append(current_frame)
            
        if en_name != '':
            # Bỏ dấu chấm than/hỏi/dấu phẩy/dấu nháy để tránh lỗi URI không hợp lệ
            clean_en_name = en_name.replace('?', '').replace('!', '').replace(',', '').replace("'", "")
            en_inst = onto.English_Idiom(clean_en_name)
            en_inst.Có_khung.append(current_frame)

print(f"Tổng số Khung đã tạo: {len(frame_dict)}")
print(f"Tổng số thành ngữ Việt: {len(onto.Vietnamese_idiom.instances())}")
print(f"Tổng số thành ngữ Anh: {len(onto.English_Idiom.instances())}")


Tổng số Khung đã tạo: 82
Tổng số thành ngữ Việt: 100
Tổng số thành ngữ Anh: 97


In [22]:
import os
# Lưu ontology ra file chuẩn v-idiomV5_final.rdf
output_dir = "../../ontology_protege"
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, "v-idiomV5_final.rdf")

onto.save(file=output_path, format="rdfxml")
print(f"Đã lưu thành công bản Ontology Gốc (không chứa suy luận) tại {output_path}")


Đã lưu thành công bản Ontology Gốc (không chứa suy luận) tại ../../ontology_protege/v-idiomV5_final.rdf


In [23]:
# Từ điển ý nghĩa chuẩn xác (tránh lỗi format từ file Markdown)
criteria_meanings = {
    # Bối cảnh
    "BC_Giao_tiếp":         "Giao tiếp, đàm phán, nói chuyện",
    "BC_Thương_mại":        "Thương mại, mua bán, kinh doanh",
    "BC_Xung_đột":          "Xung đột, cãi vã, mâu thuẫn",
    "BC_Nguy_hiểm":         "Nguy hiểm, rủi ro cao, đe dọa",
    "BC_Sinh_hoạt":         "Sinh hoạt hàng ngày, thói quen",
    "BC_Rủi_ro":            "Rủi ro, đánh cược, không chắc chắn",
    "BC_Đánh_giá":          "Đánh giá, nhìn nhận, phán xét",
    "BC_Bảo_mật":           "Bí mật, thông tin ẩn giấu",
    "BC_Sự_kiện_hiếm":      "Sự kiện hiếm gặp, bất ngờ",
    "BC_Cơ_hội":            "Cơ hội, thời cơ tốt xuất hiện",
    "BC_Cảm_xúc":           "Cảm xúc, tâm lý, nội tâm",
    "BC_Thực_thi_mục_tiêu": "Nỗ lực đạt mục tiêu",
    "BC_Hậu_quả":           "Hậu quả đã xảy ra",
    "BC_Nhận_thức":         "Nhận thức, chứng kiến, tin tưởng",
    "BC_Đối_nhân_xử_thế":   "Ứng xử xã hội giữa người với người",
    "BC_Bệnh_tật":          "Bệnh tật, sức khỏe kém",
    "BC_Cuộc_sống":         "Quy luật đời sống, thực tế xã hội",
    "BC_Gia_đình":          "Gia đình, huyết thống, cha con",
    "BC_Lao_động":          "Lao động, học tập, rèn luyện",
    "BC_Đạo_đức":           "Đạo đức, nhân cách, lòng biết ơn",
    "BC_Môi_trường_mới":    "Môi trường sống hoặc văn hóa mới",
    # Hành động
    "HD_Né_tránh":           "Né tránh, vòng vo, thoái thác",
    "HD_Đối_mặt":            "Đối mặt, dấn thân vào gian khó",
    "HD_Liều_lĩnh":          "Liều lĩnh, làm điều rủi ro",
    "HD_Tấn_công":           "Tấn công, công kích, làm hại",
    "HD_Đánh_giá_sai":       "Đánh giá sai, áp đặt chủ quan",
    "HD_Tham_lam":           "Tham lam, đòi hỏi vô lý",
    "HD_Nỗ_lực":             "Nỗ lực, làm việc chăm chỉ",
    "HD_Thổi_phồng":         "Thổi phồng, khoa trương quá đà",
    "HD_Lợi_dụng":           "Lợi dụng, vô ơn, ăn cháo đá bát",
    "HD_Che_giấu":           "Che giấu, giữ kín bí mật",
    "HD_Trì_hoãn":           "Trì hoãn, để nước đến chân mới nhảy",
    "HD_Nắm_bắt":            "Nắm bắt cơ hội ngay lập tức",
    "HD_Kiên_nhẫn":          "Kiên nhẫn, nhẫn nại chịu đựng",
    "HD_Tiêu_xài_hoang_phí": "Tiêu xài hoang phí, không tiếc",
    "HD_Kìm_nén_cảm_xúc":    "Kìm nén cảm xúc, ứng xử mềm mỏng",
    "HD_Thích_ứng":          "Thích ứng, hòa nhập môi trường mới",
    "HD_Bắt_chước":          "Bắt chước, mô phỏng thói quen",
    "HD_Đồng_lòng":          "Đồng lòng, đoàn kết, hợp lực",
    "HD_So_sánh":            "So sánh, nhìn sang hoàn cảnh người khác",
    # Kết quả
    "KQ_Thành_công":           "Thành công, đạt mục tiêu",
    "KQ_Thất_bại":             "Thất bại, hỏng việc",
    "KQ_Tổn_thất_tài_sản":     "Tổn thất tài sản, thiệt hại vật chất",
    "KQ_Tổn_thương":           "Tổn thương thể chất hoặc tinh thần",
    "KQ_Hối_hận":              "Hối hận, lời nói không cứu vãn được",
    "KQ_Lộ_tẩy":               "Lộ tẩy, bí mật bị phơi bày",
    "KQ_May_mắn":              "May mắn bất ngờ",
    "KQ_Tối_ưu_hóa":           "Tối ưu hóa, một công đôi việc",
    "KQ_Mất_quan_hệ":          "Mất quan hệ, gây thù chuốc oán",
    "KQ_Bình_yên":             "Bình yên, ổn định",
    "KQ_An_toàn":              "An toàn, thoát khỏi nguy hiểm",
    "KQ_Đạt_được_thỏa_thuận":  "Đạt được thỏa thuận, giải quyết bất đồng",
    "KQ_Kết_thúc":             "Kết thúc, mọi sự đều dừng lại",
    "KQ_Bị_khống_chế":         "Bị khống chế, kẻ ác bị trừng trị",
    "KQ_Gieo_tai_họa":         "Gieo tai họa, hậu quả thảm khốc",
    "KQ_Gắn_kết":              "Gắn kết, thắt chặt mối quan hệ",
    "KQ_Nhầm_lẫn":             "Nhầm lẫn do áp đặt chủ quan",
    "KQ_Hòa_nhập":             "Hòa nhập an toàn vào môi trường mới",
    "KQ_Tương_xứng_giá_trị":   "Tương xứng giá trị, được gì trả nấy",
    "KQ_Gặp_trở_ngại":         "Gặp trở ngại, khó khăn khi thực hiện",
    "KQ_Xác_nhận_sự_thật":     "Xác nhận sự thật, chứng kiến kiểm chứng",
    "KQ_Tích_lũy_kinh_nghiệm": "Tích lũy kinh nghiệm theo tuổi tác",
    "KQ_Tích_lũy_tri_thức":    "Tích lũy tri thức qua trải nghiệm",
    "KQ_Trân_trọng_bản_chất":  "Trân trọng phẩm chất bên trong",
    "KQ_Hình_thành_nhân_cách": "Hình thành nhân cách qua hoàn cảnh",
    "KQ_Khác_biệt_quan_điểm":  "Khác biệt quan điểm, mỗi người mỗi ý",
    "KQ_Tồn_tại_khuyết_điểm":  "Tồn tại khuyết điểm, không ai hoàn hảo",
    "KQ_Lưu_danh_hậu_thế":     "Lưu danh hậu thế, tiếng tốt lưu truyền",
    "KQ_Đối_mặt_thực_tế":      "Đối mặt thực tế cuộc sống gai góc",
    "KQ_Ngoại_cảnh_chi_phối":  "Ngoại cảnh chi phối, ngoài tầm kiểm soát",
    # Mục đích
    "MD_Bảo_vệ_bản_thân":      "Bảo vệ bản thân, giữ an toàn",
    "MD_Giải_quyết_vấn_đề":    "Giải quyết vấn đề, vượt gian khó",
    "MD_Che_đậy":              "Che đậy, giấu giếm bí mật",
    "MD_Hạ_bệ":                "Hạ bệ, đánh bại đối thủ",
    "MD_Né_tránh_trách_nhiệm": "Né tránh trách nhiệm, đổ lỗi",
    "MD_Giao_hảo":             "Giao hảo, giữ mối quan hệ tốt đẹp",
    "MD_Hòa_giải":             "Hòa giải, tha thứ người biết nhận lỗi",
    "MD_Trân_trọng_thời_gian": "Trân trọng thời gian quý báu",
    "MD_Bảo_vệ_tài_sản":       "Bảo vệ tài sản, giữ gìn thành quả",
}

# Gán ý nghĩa vào các Tiêu chí đã khởi tạo trong Ontology
with onto:
    danh_sach_classes = [onto.Bối_cảnh, onto.Hành_động, onto.Kết_quả, onto.Mục_đích]
    count = 0
    
    for cls in danh_sach_classes:
        for inst in cls.instances():
            if inst.name in criteria_meanings:
                # Xóa comment cũ đi (nếu có)
                inst.comment = [] 
                # Gán ý nghĩa mới
                inst.comment.append(criteria_meanings[inst.name])
                count += 1
                
print(f"Đã gán rdfs:comment thành công cho {count} tiêu chí.")

# Lưu đè lại Ontology ra file RDF
output_path = "../../ontology_protege/v-idiomV5_final.rdf"
onto.save(file=output_path, format="rdfxml")
print(f"Đã lưu Ontology (chứa ý nghĩa tiêu chí) thành công tại: {output_path}")


Đã gán rdfs:comment thành công cho 79 tiêu chí.
Đã lưu Ontology (chứa ý nghĩa tiêu chí) thành công tại: ../../ontology_protege/v-idiomV5_final.rdf


In [24]:
import pandas as pd
from owlready2 import locstr

# 1. Đọc file CSV v4 (đã có sẵn cả 2 cột nghĩa Tiếng Việt và Tiếng Anh)
csv_path = '../../thanh_ngu_tuc_ngu_100_cau_v4.csv'
df = pd.read_csv(csv_path)
df = df.fillna('')

# Lấy các Class
VN_cls = onto.search_one(iri="*Vietnamese_idiom")
EN_cls = onto.search_one(iri="*English_Idiom")

count_vn = 0
count_en = 0

with onto:
    for index, row in df.iterrows():
        vn_name = row['Thành_ngữ_Tiếng_Việt'].strip()
        en_name = row['Thành_ngữ_Tiếng_Anh'].strip()
        meaning_vn = row['Giải_thích_nghĩa_Tiếng_Việt'].strip()
        meaning_en = row['Giải_thích_nghĩa_Tiếng_Anh'].strip()
        
        # Gán nghĩa cho câu Tiếng Việt
        if vn_name and meaning_vn:
            try:
                vn_inst = getattr(onto, vn_name)
                if vn_inst:
                    vn_inst.comment = []
                    # Gắn chuẩn tag 'vi'
                    vn_inst.comment.append(locstr(meaning_vn, lang="vi"))
                    count_vn += 1
            except:
                pass
                
        # Gán nghĩa cho câu Tiếng Anh
        if en_name and meaning_en:
            try:
                en_inst = getattr(onto, en_name)
                if en_inst:
                    en_inst.comment = []
                    # Gắn chuẩn tag 'en'
                    en_inst.comment.append(locstr(meaning_en, lang="en"))
                    count_en += 1
            except:
                pass

# 2. Lưu đè file RDF
output_path = "../../ontology_protege/v-idiomV5_final.rdf"
onto.save(file=output_path, format="rdfxml")
print(f"Hoàn tất! Đã gán {count_vn} nghĩa Tiếng Việt (tag vi) và {count_en} nghĩa Tiếng Anh (tag en).")
print(f"Lưu file RDF thành công tại: {output_path}")


Hoàn tất! Đã gán 100 nghĩa Tiếng Việt (tag vi) và 90 nghĩa Tiếng Anh (tag en).
Lưu file RDF thành công tại: ../../ontology_protege/v-idiomV5_final.rdf
